<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/58/Uber_logo_2018.svg/1024px-Uber_logo_2018.svg.png" alt="UBER LOGO" width="50%" />

# UBER Pickups 

# import libraries

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Exploration via Kmeans

In [2]:
dataset = pd.read_csv("/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/UBER_PICKUPS/data/dataset.csv")
dataset.shape

(18707457, 8)

In [3]:
sample, reste = train_test_split(
    dataset,
    train_size=5000,
    stratify=dataset["locationid"],
    random_state=42
)

In [4]:
null = sample.isnull().sum()
print(f"Nombre de valeurs manquantes par colonne dans le dataset d'entrainement : {null.iloc[0]}")

Nombre de valeurs manquantes par colonne dans le dataset d'entrainement : 0


In [5]:
display(sample.head(1))

,date/time,lat_x,lon_x,base,weekday,locationid,borough,zone
4132132,2014-09-09 18:31:00,40.7121,-73.9344,B02682,Mardi,37,Brooklyn,Bushwick South


In [6]:
# categorical_features = ["weekday", "zone", "borough", "base"]
numerical_features = ["lat_x", "lon_x"]

In [7]:
# Preprocessing pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

In [8]:
categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

In [9]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_features),
    # ("cat", categorical_transformer, categorical_features),
])

In [10]:
X = preprocessor.fit_transform(sample)

In [11]:
# Elbow method pour déterminer le nombre optimal de clusters pour Kmeans
inertia = []
silhouette_scores = []
K = range(2, 10)


for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X, kmeans.labels_))

fig_elbow = px.line(x=K, y=inertia, title="Elbow Method for KMeans", markers=True, labels={"x": "Nombre de clusters (k)", "y": "Inertie"})
fig_elbow.show()
fig_silhouette = px.line(x=K, y=silhouette_scores, title="Silhouette Scores for KMeans", markers=True, labels={"x": "Nombre de clusters (k)", "y": "Silhouette Score"})
fig_silhouette.show()

In [12]:
X = numeric_transformer.fit_transform(sample[numerical_features])

In [13]:
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X)

sample.loc[:, "kmeans_cluster"] = kmeans.predict(X)
sample.head()

,date/time,lat_x,lon_x,base,weekday,locationid,borough,zone,kmeans_cluster
4132132,2014-09-09 18:31:00,40.712100,-73.934400,B02682,Mardi,37,Brooklyn,Bushwick South,3
5932723,2015-01-18 18:45:50,40.780491,-73.956972,B02764,Dimanche,236,Manhattan,Upper East Side North,1
17653673,2015-04-02 01:49:00,40.734611,-74.002497,B02765,Jeudi,249,Manhattan,West Village,0
5787410,2015-03-23 04:36:00,40.727944,-73.985214,B02764,Lundi,79,Manhattan,East Village,0
7395102,2015-02-21 11:57:10,40.723898,-74.001375,B02682,Samedi,211,Manhattan,SoHo,0


In [14]:
sample["kmeans_cluster"].value_counts()

kmeans_cluster
0    3188
1     950
3     670
2     192
Name: count, dtype: int64

In [15]:
map_data = sample.copy()

map_data["lat_x"] = pd.to_numeric(map_data["lat_x"], errors="coerce")
map_data["lon_x"] = pd.to_numeric(map_data["lon_x"], errors="coerce")

map_data = map_data.dropna(subset=["lat_x", "lon_x", "kmeans_cluster"])

In [16]:
cluster_colors = {
    0: "#FF6B6B",  # rouge
    1: "#19B0DE",  # bleu
    2: "#17CE48",  # vert
    3: "#F5A623",  # orange
}

In [17]:
print("Pickups in the selected sample:", len(sample))

fig = go.Figure()

for cluster_id, color in cluster_colors.items():
    df_cluster = map_data[map_data["kmeans_cluster"] == cluster_id]

    fig.add_trace(
        go.Scattermap(
            lat=df_cluster["lat_x"],
            lon=df_cluster["lon_x"],
            mode="markers",
            marker={"size": 8, "color": color},
            customdata=df_cluster[
                ["borough", "zone", "date/time", "weekday", "lat_x", "lon_x"]
            ].values,
            hovertemplate=(
                "Borough: %{customdata[0]}<br>"
                "Zone: %{customdata[1]}<br>"
                "Date: %{customdata[2]}<br>"
                "Jour: %{customdata[3]}<br>"
                "Lat: %{customdata[4]:.4f}<br>"
                "Lon: %{customdata[5]:.4f}<extra></extra>"
            ),
            name=f"Cluster {cluster_id}",
        )
    )

fig.update_layout(
    map_style="carto-positron",
    map_center={"lat": 40.73, "lon": -73.97},
    map_zoom=10,
    height=600,
    title="Prises en charge en fonction des clusters KMeans",
)

fig.show()

Pickups in the selected sample: 5000


# Db SCAN

In [18]:
Xb = sample[numerical_features]
X_scaled = StandardScaler().fit_transform(Xb)
dbscan_b = DBSCAN(eps=0.6, min_samples=100, metric="euclidean") # Le 4e cluster est considéré comme du bruit
dbscan_b.fit(X_scaled)
sample["dbscan_cluster"] = dbscan_b.labels_
sample["dbscan_cluster"].value_counts()

dbscan_cluster
 0    4515
-1     255
 2     121
 1     109
Name: count, dtype: int64

In [19]:
sample.head()

,date/time,lat_x,lon_x,base,weekday,locationid,borough,zone,kmeans_cluster,dbscan_cluster
4132132,2014-09-09 18:31:00,40.712100,-73.934400,B02682,Mardi,37,Brooklyn,Bushwick South,3,0
5932723,2015-01-18 18:45:50,40.780491,-73.956972,B02764,Dimanche,236,Manhattan,Upper East Side North,1,0
17653673,2015-04-02 01:49:00,40.734611,-74.002497,B02765,Jeudi,249,Manhattan,West Village,0,0
5787410,2015-03-23 04:36:00,40.727944,-73.985214,B02764,Lundi,79,Manhattan,East Village,0,0
7395102,2015-02-21 11:57:10,40.723898,-74.001375,B02682,Samedi,211,Manhattan,SoHo,0,0


In [20]:
map_data = sample.copy()
map_data = map_data.dropna(subset=["dbscan_cluster"])

In [21]:
cluster_colors = {
    0: "#FF6B6B",  # rouge
    1: "#19B0DE",  # bleu
    2: "#17CE48",  # vert
    3: "#F5A623",  # orange
}

In [22]:
print("Pickups in the selected sample :", len(sample))

fig = go.Figure()

for cluster_id, color in cluster_colors.items():
    df_cluster = map_data[map_data["dbscan_cluster"] == cluster_id]

    fig.add_trace(
        go.Scattermap(
            lat=df_cluster["lat_x"],
            lon=df_cluster["lon_x"],
            mode="markers",
            marker={"size": 8, "color": color},
            customdata=df_cluster[
                ["borough", "zone", "date/time", "weekday", "lat_x", "lon_x"]
            ].values,
            hovertemplate=(
                "Borough: %{customdata[0]}<br>"
                "Zone: %{customdata[1]}<br>"
                "Date: %{customdata[2]}<br>"
                "Jour: %{customdata[3]}<br>"
                "Lat: %{customdata[4]:.4f}<br>"
                "Lon: %{customdata[5]:.4f}<extra></extra>"
            ),
            name=f"Cluster {cluster_id}",
        )
    )

fig.update_layout(
    map_style="carto-positron",
    map_center={"lat": 40.73, "lon": -73.97},
    map_zoom=10,
    height=600,
    title="Prises en charge en fonction des clusters DBscan",
)

fig.show()

Pickups in the selected sample : 5000


Sur le modèle de DBsan, la méthode clustering ne parvient pas à indentifier des clusters pertinents et conserve 1 cluster qui est considéré comme du bruit. Il est donc préférable de se baser sur les clusters Kmeans pour l'analyse.

In [23]:
print("Pickups in the selected hour:", len(sample))

fig = go.Figure()

for cluster_id, color in cluster_colors.items():
    df_cluster = map_data[map_data["kmeans_cluster"] == cluster_id]

    fig.add_trace(
        go.Scattermap(
            lat=df_cluster["lat_x"],
            lon=df_cluster["lon_x"],
            mode="markers",
            marker={"size": 8, "color": color},
            customdata=df_cluster[
                ["borough", "zone", "date/time", "weekday", "lat_x", "lon_x"]
            ].values,
            hovertemplate=(
                "Borough: %{customdata[0]}<br>"
                "Zone: %{customdata[1]}<br>"
                "Date: %{customdata[2]}<br>"
                "Jour: %{customdata[3]}<br>"
                "Lat: %{customdata[4]:.4f}<br>"
                "Lon: %{customdata[5]:.4f}<extra></extra>"
            ),
            name=f"Cluster {cluster_id}",
        )
    )

fig.update_layout(
    map_style="carto-positron",
    map_center={"lat": 40.73, "lon": -73.97},
    map_zoom=10,
    height=600,
    title="Prises en charge sur 10 jours en fonction des clusters KMeans",
)

fig.show()

Pickups in the selected hour: 5000


Le modèle de Kmeans parvient a identifier les 4 clusters principaux concernant les prises en charge de clients. Le modèle semble attribuer une plus forte importance aux indications géographiques qu'aux jours de la semaine pour déterminer les clusters.

Ces résultats sont en cohérence avec ce qui était observé sur la modélisation de référence où les jours de la semaine semblait être plus déterminant dans le volume de prise en charge que dans la localisation géographique des prises en charge.